# Problem Setup: Regression

#### *Your Boss:* 
- "Should we increase TV advertising by 100$ ?"
- "How confident are you?"

In [ ]:
import pandas as pd
import plotly.express as px

# Advertising isn't bundled with ISLP; this CSV is from statlearning.com
Advertising = pd.read_csv('Advertising.csv', index_col=0)

# Interactive 3-D plot of Sales vs TV and Radio: drag to rotate, scroll to zoom, hover for values
fig = px.scatter_3d(Advertising, x='TV', y='radio', z='sales',
                    hover_data=['newspaper'], height=600)




In [ ]:
fig.update_traces(marker_size=3)
# View from the (0, 0) corner so both axes start at the front
fig.update_layout(scene_camera_eye=dict(x=-1.5, y=-1.5, z=1))
fig.show()

Now, let's view that plot from a direction perpendicular to the TV-Sales plane.

In [ ]:
# Show the plot from a direction perpendicular to the TV-Sales plane
# Look along the radio axis (from its low end), so TV runs left to right and sales runs up.
# Orthographic projection removes perspective, so the view is truly flat.
fig.update_layout(scene_camera=dict(eye=dict(x=0, y=-2.5, z=0),
                                    projection_type='orthographic'))
fig.show()


## Linear Regression: 
- Force/restrict the model to be linear: $y = mx + b$
- Which choice for $m$ and $b$ are "best"?
- How do we define/operationalize "best"?

### Three points

For simplicity, we'll work with just three stores, spread out along the `TV` axis. From here on, we only look at `sales` vs. `TV`.

In [ ]:
from regression_demos_helpers import pick_spread_points, fit_line, plot_lines, plot_residuals

# The same three stores/markets/points every run, at the 10th, 50th, and 90th percentiles of TV
pts = pick_spread_points(Advertising, 'TV', n=3)
pts[['TV', 'sales']]

### Candidate lines

Each choice of slope $m$ and intercept $b$ gives a different line. Here are three candidates. Which looks best?

In [ ]:
m_best, b_best = fit_line(pts['TV'], pts['sales'])

# Each line is just a (slope, intercept) pair
lines = {
    'Line A': (0.072, 5.4),       # passes (almost) exactly through the two outer points
    'Line B': (0.02, 12.0),
    'Line C': (m_best, b_best),   # the best-fit line; how we find it comes later
}
plot_lines(pts, lines);

### Residuals

For each point $x_i$, a line predicts $\hat{y}_i = m x_i + b$. The **residual** is the actual value minus the prediction:

$$e_i = y_i - \hat{y}_i$$

A positive residual means the point is above the line; a negative one means it's below.

In [ ]:
plot_residuals(pts, lines, 'Line A');

In [ ]:
plot_residuals(pts, lines, 'Line B');

In [ ]:
plot_residuals(pts, lines, 'Line C');

**Discuss:** Which line fits best? Line A goes right through two of the points; does that make it better than Line C? How could we combine the three residuals into one number that measures how well a line fits?

### Proposal: Whichever line has the lowest *sum* of residuals is the best.

Look at the sum of residuals for this particular, new line, line D:

In [ ]:
from regression_demos_helpers import score_table, loss_explorer

# Line D: slope -0.05, positioned to pass through the mean point (x̄, ȳ)
m_D = -0.05
b_D = pts['sales'].mean() - m_D * pts['TV'].mean()
lines['Line D'] = (m_D, b_D)
plot_residuals(pts, lines, 'Line D');

In [ ]:
# Sum of residuals for each line
score_table(pts, lines)[['sum of e']]

Line D is obviously terrible, yet its residuals sum to 0, just like Line C's. (Any line through the mean point $(\bar{x}, \bar{y})$ does this.) So the sum of residuals can't be our measure.

**Why did it fail?** Positive and negative residuals cancel.

**What we need:** every residual should add to the "badness" score, whatever its sign. Any suggestions?

<details>
<summary>Two natural options</summary>

$$\sum_i |e_i| \qquad\qquad \text{RSS} = \sum_i e_i^2$$

RSS stands for *residual sum of squares* (An unfortunate name if you ask me. I'd prefer *sum of squared residuals*, but we're stuck with RSS).

</details>

In [ ]:
score_table(pts, lines)

The choice for how to represent "badness of fit" is often called the **Loss Function**, or often just "the loss*.

**This is a choice.** The two measures can disagree: sum of $|e|$ prefers Line A, while RSS prefers Line C. Squaring punishes big misses much more than small ones, so Line A's single miss of about 6 costs it about 37 in RSS, but only costs it $|-6.09|=6.09$ in sum of |e|. 

When we measure the loss (think "badness of fit") using RSS, we call that *Least squares*  regression. This is the most common loss we'll use in the course for fitting lines to data.

### Sweeping the intercept

Fix the slope at Line C's value and slide the intercept $b$. Each slider step shifts/translates the line on the left vertically and adds one point to the loss curve on the right. The buttons switch between RSS and sum of $|e|$.

In [ ]:
import numpy as np

b_values = np.linspace(b_best - 10, b_best + 10, 41)   # 41 intercepts centered on Line C's
loss_explorer(pts, 'b', b_values, m=m_best)

**Discuss:** Where is each loss smallest? Do RSS and sum of $|e|$ pick the same intercept? Compare the shapes: RSS is a smooth bowl, while sum of $|e|$ is made of straight pieces with corners.

### Sweeping the slope

Now fix the intercept at Line C's value and slide the slope $m$ instead. The line pivots around the point where it crosses the $y$-axis.

In [ ]:
m_values = np.linspace(m_best - 0.05, m_best + 0.05, 41)   # 41 slopes centered on Line C's
loss_explorer(pts, 'm', m_values, b=b_best)

## Finding the global minimum

So far we varied one parameter at a time, holding the other fixed. But RSS depends on $m$ and $b$ *together*: it's a surface over the $(m, b)$ plane. Where is its lowest point?

In [ ]:
from regression_demos_helpers import rss_surface_3d, rss_contour

# A grid of (slope, intercept) pairs around Line C
m_grid = np.linspace(m_best - 0.06, m_best + 0.06, 101)
b_grid = np.linspace(b_best - 12, b_best + 12, 101)

# RSS for every pair on the grid; the red dot is the lowest point
rss_surface_3d(pts, m_grid, b_grid)

The same surface as a contour plot, like a topographic map: each curve connects $(m, b)$ pairs with the same RSS. Every candidate line is a single point on this map.

In [ ]:
rss_contour(pts, m_grid, b_grid, lines);   # Line D is far off this map

### A formula for the best line

Using calculus (set both partial derivatives of RSS to zero; we'll skip the details), the minimum is at

$$\hat{m} = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2}, \qquad \hat{b} = \bar{y} - \hat{m}\,\bar{x}$$

The book writes these as $\hat{\beta}_1$ (slope) and $\hat{\beta}_0$ (intercept). Hardly anyone has them memorized, and you don't need to. The point is that there's a *formula* for the best slope and intercept, so no searching is needed.

In [ ]:
x, y = pts['TV'], pts['sales']

m_hat = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b_hat = y.mean() - m_hat * x.mean()
print(f'slope = {m_hat:.4f}, intercept = {b_hat:.4f}')

In practice, you call a library that does this for you. 
Here are a a couple of examples:

In [ ]:
from sklearn.linear_model import LinearRegression 

# NumPy: fit a degree-1 polynomial; returns [slope, intercept]
m_np, b_np = np.polyfit(x, y, deg=1)

# scikit-learn: X must be 2-D (a DataFrame with one column)
sk_model = LinearRegression().fit(pts[['TV']], y)


print(f'numpy:        slope = {m_np:.4f}, intercept = {b_np:.4f}')
print(f'scikit-learn: slope = {sk_model.coef_[0]:.4f}, intercept = {sk_model.intercept_:.4f}')

## Should we spend more on TV advertising?

In [ ]:
# Use Line C, the least squares line through our 3 stores (m_hat, b_hat from above)

# Predict sales corresponding to current max spending on TV
x_max = Advertising['TV'].max()
y_max = m_hat * x_max + b_hat

# Predict sales for 100$ more than current max spending on TV
x_new = x_max + 100
y_new = m_hat * x_new + b_hat

# Compare the two predictions
print(f"Current sales: {y_max:.2f} units")
print(f"Predicted sales for 100k$ more in TV spending: {y_new:.2f} units$")

# Print percentage increase
percentage_increase = ((y_new - y_max) / y_max) * 100
print(f"Percentage increase: {percentage_increase:.2f}%")


Using a line based on a particular 3-store sample, an estimated 30 percent increase is impressive (depending on how much each unit costs, sells for), but **how confident** should we be in our prediction?

## Formalizing Confidence

Three questions we'd like to answer about a fitted line:

1. **$R^2$:** How much better does the model predict the data than just predicting the mean? Is that enough to quantify confidence?
2. **p-value:** If there were really no relationship, how likely is it that a random sample would still produce an $R^2$ this big (or bigger)?
3. **Standard error:** How precisely did we estimate the slope and intercept? Can we put error bars on them, or give a range of values we're 95% confident about?

### $R^2$: better than guessing the mean?

The simplest possible model ignores `TV` and always predicts the average sales, $\bar{y}$. Compare its residuals to Line C's:

In [ ]:
from matplotlib.pyplot import subplots
from regression_demos_helpers import r_squared, plot_scenario

# Line C vs. a flat line at the mean of sales (slope 0)
compare = {'Line C (best fit)': (m_best, b_best),
           'Mean only': (0, pts['sales'].mean())}

fig, axes = subplots(1, 2, figsize=(14, 5), sharey=True)
plot_residuals(pts, compare, 'Line C (best fit)', ax=axes[0], show_rss=True)
plot_residuals(pts, compare, 'Mean only', ax=axes[1], show_rss=True);

**How much better is the model than just predicting the average?**

The mean-only line's sum of squares has its own name, the **total sum of squares** (TSS). It measures how much $y$ varies in the first place. It's tempting to just subtract, TSS − RSS. But that number depends on the units of $y$:

In [ ]:
x, y = pts['TV'], pts['sales']   # sales is recorded in thousands of units

# Same data and same fit, with sales measured two ways
for scale, unit in [(1, 'thousands of units'), (1000, '(plane/ordinary) units')]:
    y_scaled = y * scale
    m, b = fit_line(x, y_scaled)
    rss = np.sum((y_scaled - (m * x + b)) ** 2)
    tss = np.sum((y_scaled - y_scaled.mean()) ** 2)
    print(f'sales in {unit:>18}:  TSS - RSS = {tss - rss:>14,.1f}   '
          f'(TSS - RSS) / TSS = {(tss - rss) / tss:.3f}')

Changing units multiplies the difference by a million, even though the fit is exactly as good. The *fraction* doesn't change, and that fraction is $R^2$:

$$R^2 = \frac{\text{TSS} - \text{RSS}}{\text{TSS}} = 1 - \frac{\text{RSS}}{\text{TSS}}$$

It's the fraction of the variation in $y$ that the model explains (the "variance explained").

<details> 
<summary>What R-squared value indicates a perfect fit? And what about a terrible fit?</summary>

**Perfect fit:** $\text{RSS}=0$ and $\text{TSS} > 0$
plug into equation for $R^2$, and get $R^2=1$.

In practice, for a great fit, we'll have $\text{RSS}$, slightly greater than $0$ and $\text{TSS} > 0$ so, $R^2$ will be slightly less than $1$.

**Terrible fit:** $\text{RSS} \approx \text{TSS}$ (The model predicts y-values no better than $\bar{y}$)
plug into equation for $R^2$, and get $R^2=0$.

In practice, terrible fits will typically have $R^2$ values slightly greater than $0$.

</details>

<details>
<summary> Terminology Gotcha:</summary>

Given that, $R^2 = 1 - \frac{\text{RSS}}{\text{TSS}}$

...then is this true?

$$R = \sqrt{ 1 - \frac{\text{RSS}}{\text{TSS}}} \text{    <- This is NOT true!!!}$$

The answer is **No**! 

"$R^2$" is *not* a mathematical expression; It is just a *name*. We refer to the statistic above as "R-squared" and we often write that name like this: "$R^2$. This confusing (and I think *terrible*) name is an unfortunate accident of history and for now, we're stuck with it.
</details>

<details>
<summary>R-Sqaured as Fraction of Variance Explained</summary>

**Purpose:** 

Intuitively, the $y$ values of our data have some kind of average spread or variation (the variance in $y$). Our *model* can account for or "explain" a certain amount of that spread or variation. $R^2$ is a measure of how well (or poorly) our model "explains" that variation.

**Variance** measures how spread out a set of numbers is: the average squared distance from their mean.

$$\text{var}(z) = \frac{1}{n}\sum_{i=1}^{n}(z_i - \bar{z})^2$$

**Step 1: split the fraction.**

$$R^2 = \frac{\text{TSS} - \text{RSS}}{\text{TSS}} = \frac{\text{TSS}}{\text{TSS}} - \frac{\text{RSS}}{\text{TSS}}$$

**Step 2: divide each numerator and denominator by $n$**, the number of residuals. This doesn't change either fraction.

$$R^2 = \frac{\text{TSS}/n}{\text{TSS}/n} - \frac{\text{RSS}/n}{\text{TSS}/n}$$

**Step 3: turn each $(\ldots)/n$ into a variance.** We have two sets of residuals:

- Predicting with $\bar{y}$: $\; e_i^{\bar{y}} = y_i - \bar{y}$, so $\text{TSS} = \sum_i \left(e_i^{\bar{y}}\right)^2$
- Predicting with the model: $\; e_i^{\text{model}} = y_i - \hat{y}_i$, so $\text{RSS} = \sum_i \left(e_i^{\text{model}}\right)^2$

Both sets of residuals have a mean of 0 (for the least squares line, that's the $\sum_i e_i = 0$ fact from earlier). With $\bar{e} = 0$, a sum of squared residuals divided by $n$ is exactly their variance:

$$\frac{\text{TSS}}{n} = \frac{1}{n}\sum_i \left(e_i^{\bar{y}} - 0\right)^2 = \text{var}(e^{\bar{y}}) \qquad\qquad \frac{\text{RSS}}{n} = \text{var}(e^{\text{model}})$$

Substituting:

$$R^2 = \frac{\text{var}(e^{\bar{y}})}{\text{var}(e^{\bar{y}})} - \frac{\text{var}(e^{\text{model}})}{\text{var}(e^{\bar{y}})}$$

**The first term is the whole.** $\text{var}(e^{\bar{y}})$ is all of the variance in the residuals when we make the naive prediction $\bar{y}$. Divided by itself, it's 1: 100% of the variance in $e^{\bar{y}}$. Notice that $\text{var}(e^{\bar{y}})$ is also just $\text{var}(y)$ by the definition of variance.

**The second term is the fraction left over.** It's the fraction of that whole (the variance in $y$) that still remains in the residuals when we compute residuals using the model.

- If $\frac{\text{var}(e^{\text{model}})}{\text{var}(e^{\bar{y}})} = 0.01$, the model fits the data very well: its residuals have only 1% of the variance of the $\bar{y}$ residuals. Then $R^2 = 0.99$.
- If $\frac{\text{var}(e^{\text{model}})}{\text{var}(e^{\bar{y}})} \approx 0.9$, computing the residuals with the model gives a spread very similar to computing them with $\bar{y}$. The model barely beats guessing the mean, and $R^2 \approx 0.1$.

**Putting it together:**

$$R^2 = 1 - \frac{\text{var}(e^{\text{model}})}{\text{var}(e^{\bar{y}})}$$

is the fraction by which the model *reduced* the variance in the residuals, relative to the naive prediction $\bar{y}$. So, $R^2$ can be interpreted as variance in $y$ explained by the model. In the jargon of statistics (and in ISL, our textbook), this is the fraction of **variance explained** by the model.

**Conclusion:**
Intuitively, the $y$ values have some kind of average spread or variation (the variance in $y$). Our *model* can account for or "explain" a certain amount of that spread or variation. $R^2$ is a measure of how well (or poorly) our model "explains" that variation.

</details>

In [ ]:
print(f'R² for Line C on our three stores: {r_squared(x, y):.3f}')

### A Few Cases 

**We're leaving the Advertising data now.** The next examples use simulated data, so we know the true function behind each one (the dotted gray curve). First, a baseline where a straight line fits well:

A **pretty good fit** to the sample, so a pretty good $\text{R}^2$

In [ ]:
# Simulated data (not Advertising), generated with a fixed seed in regression_demos_helpers.make_scenario
plot_scenario('baseline');

**Noisy $y$.** Same true line, but with much more variation that $x$ can't explain. The fitted line is still close to the truth, yet $R^2$ is low.

In [ ]:
plot_scenario('noisy');

**Wrong functional form.** The truth is a curve (a parabola plus a little noise). No straight line can follow it.

In [ ]:
plot_scenario('curved');

**Narrow range of $x$.** Same true line and noise level as the baseline, but every $x$ falls between 4.5 and 5.5. Over such a short range, the noise swamps the trend.

In [ ]:
plot_scenario('narrow');

**Outliers in $y$.** The baseline again, with three points pushed far upward. A few extreme points inflate RSS and translate or tilt the line.

In [ ]:
plot_scenario('outliers');

## p-Value

**Is $R^2$ enough to quantify confidence?**

In the below plots, pretend we don't have all the data. We draw a sample of 3 points and fit a line to each sample set.

<details>
<summary>Answer:</summary>

No. R^2 depends on the sample.

- A low $R^2$ can mean a real but noisy relationship, the wrong shape of model, too narrow a range of data, or a few outliers. 

- And a high $R^2$ from only three points (the beginning of this section), could just be due to a "unlucky" sample. Unlucky, because if we rely on $R^2$ inappropriately, we have false confidence in our model's ability to make predictions, when really, it just happens to do a good job making predictions on the data *we happened to pick*. The next two topics (p-values and standard errors) address this.

</details>

In [ ]:
# Back to the real Advertising data: draw 1000 random samples of 3 stores and fit a line to each.
# Shown: five of those samples, from the lowest R² to the highest, then one more picked at random.
from regression_demos_helpers import plot_sample_luck

plot_sample_luck(Advertising, n=3, draws=1000);

Line C is the least squares line through our three stores, with $R^2 \approx 0.85$. That looks good, but three points is very little data. **If TV had no relationship to sales at all, how often would three random stores give an $R^2$ at least this large just by luck?**

**Breaking the relationship on purpose.** To see what luck alone can do, we build samples where `TV` and `sales` have nothing to do with each other: take the TV budgets of 3 random stores and the sales of 3 *other* random stores, and pair them up. Any pattern in such a sample is pure coincidence.

In [ ]:
# Real Advertising values, but each sample pairs TV from 3 random stores with sales from 3 other stores
from regression_demos_helpers import plot_null_samples

plot_null_samples(Advertising, n=3, k=3);

Even with no relationship, some samples produce a steep line and a high $R^2$. How often? Let's draw 1000 of these samples and build up a histogram of their $R^2$ values. Press **Play** or drag the slider: the first 10 draws are added one at a time, then in bigger jumps.

In [ ]:
from regression_demos_helpers import null_r2_explorer

null_r2_explorer(Advertising, n=3, draws=1000)

Now compare Line C against this pile of luck-only values. What fraction of the "no relationship" samples did at least as well as Line C?

In [ ]:
from regression_demos_helpers import null_r2, plot_r2_histogram

r2_C = r_squared(pts['TV'], pts['sales'])
_, null_values = null_r2(Advertising, n=3, draws=1000)   # same 1000 draws as the slider above
p_simulated = np.mean(null_values >= r2_C)

plot_r2_histogram(null_values, shade_above=r2_C)
print(f'Fraction of no-relationship samples with R² ≥ {r2_C:.3f}: {p_simulated:.3f}')

### The p-value

That fraction (0.266 in the plot) is called the **p-value**: *if there were no relationship, between TV and Sales, how often would random chance give an $R^2$ at least as good as our model's $R^2$?*

Here, roughly a quarter of the "no relationship" samples of 3 stores did as well as Line C. Despite its "good" $R^2$, Line C is not convincing evidence that TV spending is related to sales.

- A **small** p-value (a common cutoff is 0.05) means our result would be very surprising if there were no relationship, so we have evidence that there is one.
- The p-value is *not* the probability that the model is wrong, nor the probability that there's no relationship. It only measures how surprising a fit this good (an $R^2$ at least this large) would be if there were no relationship.

In practice, nobody simulates this by hand. Libraries compute the p-value from a formula (which assumes the residuals are roughly normally distributed), so expect a close match, not an identical one:

In [ ]:
from scipy.stats import linregress

# fit_line (and np.polyfit) only return the slope and intercept, so refit with scipy to get the p-value
fit_C = linregress(pts['TV'], pts['sales'])
print(f'slope: {fit_C.slope:.4f} (Line C: {m_best:.4f})')
print(f'simulated p-value: {p_simulated:.3f}')
print(f'scipy p-value:     {fit_C.pvalue:.3f}')

### What if we had more data?

Repeat the whole experiment with 30 stores instead of 3: take a real random sample of 30 stores and fit a line, then compare its $R^2$ to 1000 "no relationship" samples of 30.

In [ ]:
stores_30 = Advertising.sample(30, random_state=463)   # a real sample: TV and sales from the same stores
r2_30 = r_squared(stores_30['TV'], stores_30['sales'])
_, null_30 = null_r2(Advertising, n=30, draws=1000)
fit_30 = linregress(stores_30['TV'], stores_30['sales'])

plot_r2_histogram(null_30, shade_above=r2_30)
print(f'R² for 30 real stores: {r2_30:.3f}')
print(f'No-relationship samples at least that good: {np.sum(null_30 >= r2_30)} of 1000')
print(f'scipy p-value: {fit_30.pvalue:.2g}')

Remember, the p-value is the blue area to the right of the vertical dashed line divided by the total blue area. That's basically p-value = 0/(a big number) = 0.

With 30 random, no-relationship samples:
- Luck alone almost never produces a large $R^2$
- The "no relationship" histogram piles $R^2$ values near 0
- The real sample's $R^2$ sits far out in the tail (vertical red, dashed line). 
- None of the 1000 simulated samples reach $R^2=0.75$, so the p-value is essentially 0. **The same kind of fit is far more convincing with more data.**

### Standard error

The p-value asked whether there's a relationship at all. The third question asks **how precisely** we've pinned down the slope (or other model parameters): if we collected a new sample, how much would our estimate be likely to change?

**We're leaving the real data again.** To measure how much the slope estimate bounces from sample to sample, we need to know the true line the samples come from, so in this next section we simulate noisy data so we have the formula for that true line.

<details>
<summary>How this simulation works (optional)</summary>

- **A pretend true line.** With real data we never know the true line, so we *treat* the least squares line through all 200 Advertising stores as though it were the truth.
- **Noise of a realistic size.** Each simulated store's sales are the true line's prediction plus fresh random noise (centered on 0). The noise's typical size matches how far the real stores sit from that line (their residual standard error, printed below as `noise sd`). We don't reuse the real residuals, just their spread.
- **Same TV budgets every time.** Every simulated sample uses the same 30 TV budgets; only the noise (and so the sales) changes.
- **A simplification.** The simulated noise has the same spread at every TV value, while the real residuals fan out as TV grows. That doesn't matter for this demo.

</details>

Press **Play**: each new sample gives a slightly different fitted line and a slightly different slope.

In [ ]:
# Simulated data: a known true line, with its numbers borrowed from all 200 Advertising stores
from regression_demos_helpers import make_truth, slope_explorer

truth = make_truth(Advertising, n=30)   # true slope and intercept, noise sd, and 30 fixed TV budgets
print(f"true line: sales = {truth['m']:.4f} TV + {truth['b']:.2f},   noise sd = {truth['sigma']:.2f}")

slope_explorer(truth, draws=1000)

The **standard error (SE)** of the slope is the typical size of this bounce: the standard deviation of that histogram. It tells you how much your estimate would change if you collected a new sample.

Notice the histogram is centered on the true slope: the estimates are right *on average*, but any single one typically misses by about one SE.

**In real life, you have one sample, not 1000.** But the software can estimate the bounce from that one sample alone (scipy calls it `stderr`; regression tables label it `std err`). Compare the SE from just the first simulated sample (orange bar) with the SD of all 1000 estimates (shaded band):

In [ ]:
from regression_demos_helpers import simulate_fits, plot_slope_histogram

ys, fits = simulate_fits(truth, draws=1000)   # the same 1000 samples as the slider above
one_se = fits['stderr'][0]

plot_slope_histogram(fits['slope'], truth['m'], one_se=one_se)
print(f"SD of 1000 slope estimates (needs the simulation): {fits['slope'].std():.4f}")
print(f"SE from sample 1 alone (scipy's stderr):            {one_se:.4f}")

Close, from a single sample and no simulation. (Not identical: the SE is itself estimated from the sample, so it bounces a little too.)

### Using the standard error

**Confidence interval.** Estimate ± 2 SE gives a range of plausible values for the true slope. Does it work? Here are the intervals from the first 50 simulated samples:

In [ ]:
from regression_demos_helpers import plot_intervals

plot_intervals(fits, truth['m'], k=50)

low, high = fits['slope'] - 2 * fits['stderr'], fits['slope'] + 2 * fits['stderr']
covered = ((low <= truth['m']) & (truth['m'] <= high)).mean()
print(f'Fraction of all 1000 intervals that contain the true slope: {covered:.3f}')

About 95% of the intervals contain the true slope, so estimate ± 2 SE is called a **95% confidence interval**. (Any single interval either contains the truth or doesn't; the 95% describes how often the *method* works.)

**t-statistic.** Estimate ÷ SE counts how many "bounces" the estimate is from zero. If it's far (say, more than 2), a true slope of zero is implausible. This is how software computes the p-value from the previous section: a big t means a small p-value.

<details>
<summary>Optional: what makes the SE smaller?</summary>

Less noise, more data, and more spread in $x$. Each panel below compares the baseline with one change. Halving the noise or using 4× as many stores cuts the SE in half; squeezing the TV budgets to half their spread doubles it.

</details>

In [ ]:
# Optional: still simulated, from the same true line
from regression_demos_helpers import plot_se_factors

plot_se_factors(truth);

### Reading a regression table

**Back to the real data**, all 200 stores. This is the kind of table regression software prints (compare ISL Table 3.1):

In [ ]:
from scipy.stats import linregress

fit_all = linregress(Advertising['TV'], Advertising['sales'])
table = pd.DataFrame({'coef': [fit_all.intercept, fit_all.slope],
                      'std err': [fit_all.intercept_stderr, fit_all.stderr]},
                     index=['intercept', 'TV'])
table['t'] = table['coef'] / table['std err']
table['95% CI low'] = table['coef'] - 2 * table['std err']
table['95% CI high'] = table['coef'] + 2 * table['std err']
table.round(4)

In the table above, "intercept" refer's to the model's y-intercept and `TV` refer's to the model's estimate of the coefficient of the `TV` variable. In other words, `TV` refers to the slope of our linear model.

How to read the `TV` row: **the slope is 0.0475 with standard error 0.0027, so the true slope is probably between about 0.042 and 0.053, and it's clearly not zero** (t ≈ 17.7: the estimate is almost 18 standard errors away from zero, so the p-value is essentially 0).

# Putting it All Together: Should We Increase TV Advertising?

Back to the boss's questions from the very top of the notebook:

- "Should we increase TV advertising by $100?"
- "How confident are you?"

**Units first.** In this dataset, `TV` is in thousands of dollars and `sales` is in thousands of units. So "+100" in the data means **+$100,000** of TV advertising, and a slope of 0.0475 means about 47.5 more units sold per extra $1,000. (ISL describes each row as a *market*; we've been calling them stores.)

**The increase is slope × change.** For a straight line, raising TV by 100 raises predicted sales by slope × 100, no matter what budget we start from; the intercept cancels out. So an interval for the slope turns into an interval for the increase: just multiply by 100.

Here's the answer from our 3 stores (the earlier "30% increase") next to the answer from all 200 markets:

In [ ]:
from scipy.stats import linregress, t as t_dist

extra_tv = 100   # TV is in thousands of dollars, so this is +$100,000

rows = {}
for label, data in [('3 stores (Line C)', pts), ('all 200 markets', Advertising)]:
    fit = linregress(data['TV'], data['sales'])
    multiplier = t_dist.ppf(0.975, len(data) - 2)   # how many SEs a 95% interval needs (≈ 2 with lots of data)
    rows[label] = {'slope': fit.slope, 'SE': fit.stderr, 'p-value': fit.pvalue,
                   'SEs for 95%': multiplier,
                   'increase (thousand units)': extra_tv * fit.slope,
                   'low': extra_tv * (fit.slope - multiplier * fit.stderr),
                   'high': extra_tv * (fit.slope + multiplier * fit.stderr)}

comparison = pd.DataFrame(rows).T
comparison.round(4)

Same method, very different confidence:

- **3 stores:** about 7 thousand more units, but the 95% interval runs from a large *drop* in sales to a huge gain. It includes 0, which matches the p-value of about 0.25: we can't even rule out "no effect."
- **All 200 markets:** about 4.75 thousand more units, probably between about 4.2 and 5.3 thousand.

<details>
<summary>Why does the 3-store interval use about 12.7 SEs instead of 2?</summary>

"Estimate ± 2 SE" works when there's a decent amount of data. With very few points, the SE is itself a shaky estimate (it's based on only one leftover residual here), so a 95% interval has to be much wider to be safe. The `SEs for 95%` column shows the right multiplier: about 12.7 for 3 points, and about 1.97 (≈ 2) for 200.

</details>

Now let's see the prediction itself: sales at the largest TV budget in our data plus $100k.

In [ ]:
from regression_demos_helpers import plot_tv_decision

plot_tv_decision(Advertising, pts, extra=extra_tv);

The shaded band shows where the *average* sales at each budget probably lies (individual markets scatter much more widely). It's narrowest in the middle of our data and widens as we move away, and our prediction sits out in the "beyond our data" region.

### How confident, honestly?

The standard error only measures sampling luck, *assuming the straight-line model is right*. It doesn't cover:

- **Extrapolation.** Largest budget + $100k is beyond every budget in our data. We have no evidence the line keeps going straight out there; returns might level off.
- **Association, not causation.** This is observational data. Markets that spend a lot on TV might differ in other ways (bigger cities, more stores) that also raise sales.
- **Other media.** Radio and newspaper spending also affect sales, and our model ignores them. That's where multiple regression comes in, next time.
- **The spread isn't constant.** The residuals fan out as TV grows, so the standard textbook SEs are a bit optimistic.

### Your turn: answer the boss

In 2–3 sentences, answer both of the boss's questions: should we spend $100k more on TV, and how confident are you?

<details>
<summary>One possible answer</summary>

"Within the budgets we've seen, each extra $1,000 of TV advertising goes with about 42 to 53 more units sold. An extra $100k would suggest roughly 4,200 to 5,300 more units, but that's beyond any budget in our data, so treat it as a rough guess. Also, our data show an association, not proof that TV advertising causes the extra sales."

</details>